# Stage 3.2 v2: Cleaning and freezing my final retrieval evidence

**Project:** Agentic AI system for automated research and report generation  
**Application topic:** AI in healthcare  
**Student:** Adarsh Konderu

A pre-experiment audit found publication-system text at the end of two PMC records. I preserved the original corpus and created a deterministic cleaning audit. This notebook now:

1. verifies the cleaned 46-article corpus and cleaning record;
2. rebuilds all chunks and embeddings in one fixed software environment;
3. reuses the complete tested Stage 3.1 hybrid retrieval implementation;
4. screens 15 candidate evaluation questions for corpus answerability; and
5. saves a new evidence package with checksums.

The inclusion set remains 46 CC BY articles. This stage removes only confirmed text-extraction artefacts and does not add or exclude research papers.

## 1. Why I am implementing retrieval directly

I am using plain Python and Sentence Transformers rather than LangChain at this
stage. This makes the main RAG operations visible and easier for me to explain:

`question → question embedding → similarity comparison → relevant chunks`

The model used here creates embeddings only. It is not an LLM judge and it does
not write the final answer.


In [ ]:
# Stage 4 uses Sentence Transformers 3.4.1, so this stage uses the same
# version to prevent document and question embeddings being made by
# different library releases.
!pip -q install "transformers==4.49.0" "sentence-transformers==3.4.1"

## 2. Import libraries and record the frozen settings

The cleaning rules, chunk size, embedding revision and hybrid retrieval values are fixed before the final reports are generated.

In [ ]:
import csv
import hashlib
import importlib.metadata
import io
import json
import math
import re
import shutil
import sys
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from google.colab import files
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/content/stage_3_2_input")
OUTPUT_DIR = Path("/content/stage_3_2_outputs_v2")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_ARTICLES = 46
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
CHUNK_SIZE_WORDS = 220
CHUNK_OVERLAP_WORDS = 40
MINIMUM_FINAL_CHUNK_WORDS = 80
TOP_K = 5

# These are the selected Stage 3.1 hybrid-retrieval settings.
MAXIMUM_PER_ARTICLE = 1
RRF_CONSTANT = 60
DENSE_WEIGHT = 0.65
BM25_WEIGHT = 0.35
BM25_K1 = 1.5
BM25_B = 0.75

print("Embedding model:", MODEL_NAME)
print("Embedding revision:", MODEL_REVISION)
print("Chunk size and overlap:", CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS)
print("Hybrid weights:", DENSE_WEIGHT, BM25_WEIGHT)

## 3. Upload and verify the cleaned corpus package

Upload `Adarsh_Konderu_Stage_3_2_Clean_Corpus_Input.zip`. The package contains the cleaned JSONL corpus, the record-level audit, the cleaning rules and checksums.

In [ ]:
uploaded_files = files.upload()
expected_name = "Adarsh_Konderu_Stage_3_2_Clean_Corpus_Input.zip"
if expected_name in uploaded_files:
    uploaded_name = expected_name
elif len(uploaded_files) == 1:
    uploaded_name = next(iter(uploaded_files))
else:
    raise ValueError("Please upload only the Stage 3.2 clean-corpus ZIP.")

with zipfile.ZipFile(io.BytesIO(uploaded_files[uploaded_name])) as archive:
    for member in archive.namelist():
        member_path = Path(member)
        if member_path.is_absolute() or ".." in member_path.parts:
            raise ValueError(f"Unsafe archive member: {member}")
    archive.extractall(INPUT_DIR)

required_files = [
    "pmc_corpus_cleaned.jsonl",
    "corpus_cleaning_audit.csv",
    "corpus_cleaning_rules.json",
    "checksums_sha256.json",
]
for required_file in required_files:
    assert (INPUT_DIR / required_file).exists(), f"Missing {required_file}"

expected_checksums = json.loads(
    (INPUT_DIR / "checksums_sha256.json").read_text(encoding="utf-8")
)
for file_name, expected_digest in expected_checksums.items():
    actual_digest = hashlib.sha256((INPUT_DIR / file_name).read_bytes()).hexdigest()
    assert actual_digest == expected_digest, f"Checksum mismatch: {file_name}"

corpus_path = INPUT_DIR / "pmc_corpus_cleaned.jsonl"
records = [
    json.loads(line)
    for line in corpus_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
pmcids = [record["pmcid"] for record in records]
assert len(records) == EXPECTED_ARTICLES
assert len(set(pmcids)) == EXPECTED_ARTICLES
assert all(record["licence"] == "CC BY" for record in records)
assert all(record.get("full_text", "").strip() for record in records)
assert sum(bool(record.get("cleaning_applied")) for record in records) == 2

forbidden_fragments = [
    "-->PDIG-D-26-00413",
    "Additional Editor Comments (if provided)",
    "Peer Review File Supplementary Information",
]
assert not any(
    fragment.lower() in record["full_text"].lower()
    for record in records
    for fragment in forbidden_fragments
), "Publication-system text remains in the corpus"

print("Clean corpus verified")
print("Articles:", len(records))
print("Cleaned records:", sum(bool(r.get("cleaning_applied")) for r in records))
print("Themes:", dict(Counter(record["theme"] for record in records)))

## 4. Split each article into overlapping chunks

A complete article is too large and too general to retrieve as a single item.
I therefore use 220-word chunks with a 40-word overlap. The overlap reduces the
chance of losing an important idea at the boundary between two chunks.

The final part of an article is joined to the preceding chunk when it would be
shorter than 80 words. This avoids very small, low-context chunks.


In [ ]:
def clean_text(text):
    """Remove repeated whitespace while keeping the article wording unchanged."""
    return re.sub(r"\s+", " ", text).strip()


def split_into_word_chunks(text, chunk_size, overlap, minimum_final_size):
    """Split text into fixed word windows with a controlled overlap."""
    words = clean_text(text).split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))

        # Absorb a short remainder so no article ends with a tiny chunk.
        if 0 < len(words) - end < minimum_final_size:
            end = len(words)

        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break
        start = end - overlap

    return chunks


def build_chunks(article_records):
    """Create traceable chunks while retaining citation metadata."""
    chunk_records = []

    for article in article_records:
        article_chunks = split_into_word_chunks(
            article["full_text"],
            CHUNK_SIZE_WORDS,
            CHUNK_OVERLAP_WORDS,
            MINIMUM_FINAL_CHUNK_WORDS,
        )

        for position, chunk_text in enumerate(article_chunks, start=1):
            chunk_records.append(
                {
                    "chunk_id": f'{article["pmcid"]}_C{position:04d}',
                    "pmcid": article["pmcid"],
                    "doi": article["doi"],
                    "title": article["title"],
                    "authors": article["authors"],
                    "year": article["year"],
                    "theme": article["theme"],
                    "source_url": article["source_url"],
                    "chunk_position": position,
                    "chunk_word_count": len(chunk_text.split()),
                    "text": chunk_text,
                }
            )

    return chunk_records


In [ ]:
chunks = build_chunks(records)

# Integrity checks connect every chunk back to one of the 46 articles.
chunk_ids = [chunk["chunk_id"] for chunk in chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Duplicate chunk ID found"
assert {chunk["pmcid"] for chunk in chunks} == set(pmcids), "Article lost during chunking"
assert all(chunk["chunk_word_count"] >= MINIMUM_FINAL_CHUNK_WORDS for chunk in chunks)

chunk_counts = Counter(chunk["pmcid"] for chunk in chunks)
print("Chunks created:", len(chunks))
print("Articles represented:", len(chunk_counts))
print("Minimum chunks from one article:", min(chunk_counts.values()))
print("Maximum chunks from one article:", max(chunk_counts.values()))


## 5. Rebuild the embeddings in the frozen environment

The model revision and Sentence Transformers version now match the environment used for query embeddings in the final Agentic RAG system.

In [ ]:
embedding_model = SentenceTransformer(
    MODEL_NAME,
    revision=MODEL_REVISION,
)

texts_for_embedding = [
    f'Title: {chunk["title"]}\nTheme: {chunk["theme"]}\nText: {chunk["text"]}'
    for chunk in chunks
]
embeddings = embedding_model.encode(
    texts_for_embedding,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
assert embeddings.shape == (len(chunks), 384)
assert np.isfinite(embeddings).all()
print("Embedding matrix shape:", embeddings.shape)

## 6. Rebuild the selected hybrid retriever

Dense similarity captures meaning and BM25 rewards exact terms. Weighted Reciprocal Rank Fusion combines the two rank positions. Only one chunk per article can enter the first five results, which prevents one long paper from dominating the evidence.

In [ ]:
# This tokenisation is copied from the tested Stage 3.1 hybrid retriever.
# Keeping it identical prevents a hidden change to the BM25 rankings.
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being", "by",
    "can", "could", "did", "do", "does", "for", "from", "had", "has",
    "have", "how", "in", "into", "is", "it", "its", "may", "of", "on",
    "or", "should", "that", "the", "their", "these", "this", "to", "use",
    "used", "using", "was", "were", "what", "when", "where", "which",
    "with", "would",
}
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)?")


def tokenise_for_bm25(text):
    """Create the lower-case BM25 terms used in Stage 3.1."""
    return [
        token
        for token in TOKEN_PATTERN.findall(text.lower())
        if token not in STOP_WORDS and len(token) >= 2
    ]


chunk_tokens = [
    tokenise_for_bm25(
        f'{chunk["title"]} {chunk["theme"].replace("_", " ")} '
        f'{chunk["text"]}'
    )
    for chunk in chunks
]
document_lengths = np.array(
    [len(tokens) for tokens in chunk_tokens], dtype=np.float32
)
average_document_length = float(document_lengths.mean())
term_frequencies = [Counter(tokens) for tokens in chunk_tokens]

document_frequencies = Counter()
for frequencies in term_frequencies:
    document_frequencies.update(frequencies.keys())

document_count = len(chunks)
inverse_document_frequency = {
    term: math.log(
        1 + (document_count - frequency + 0.5) / (frequency + 0.5)
    )
    for term, frequency in document_frequencies.items()
}


def calculate_bm25_scores(question):
    """Return one BM25 score for every stored chunk."""
    query_terms = tokenise_for_bm25(question)
    scores = np.zeros(document_count, dtype=np.float32)
    for index, frequencies in enumerate(term_frequencies):
        length_adjustment = BM25_K1 * (
            1 - BM25_B
            + BM25_B * document_lengths[index] / average_document_length
        )
        score = 0.0
        for term in query_terms:
            frequency = frequencies.get(term, 0)
            if frequency == 0:
                continue
            score += inverse_document_frequency.get(term, 0.0) * (
                frequency * (BM25_K1 + 1)
                / (frequency + length_adjustment)
            )
        scores[index] = score
    return scores


def rank_positions(scores):
    """Convert scores into one-based rank positions."""
    order = np.argsort(-scores)
    positions = np.empty(len(order), dtype=int)
    positions[order] = np.arange(1, len(order) + 1)
    return positions


def hybrid_retrieve(question, top_k=TOP_K):
    """Retrieve evidence using the frozen Stage 3.1 method."""
    if not question.strip():
        raise ValueError("Question cannot be empty")
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]
    dense_scores = embeddings @ question_embedding
    bm25_scores = calculate_bm25_scores(question)
    dense_ranks = rank_positions(dense_scores)
    bm25_ranks = rank_positions(bm25_scores)
    hybrid_scores = (
        DENSE_WEIGHT / (RRF_CONSTANT + dense_ranks)
        + BM25_WEIGHT / (RRF_CONSTANT + bm25_ranks)
    )
    order = np.argsort(-hybrid_scores)
    selected = []
    counts = Counter()
    for index in order:
        chunk = chunks[int(index)]
        if counts[chunk["pmcid"]] >= MAXIMUM_PER_ARTICLE:
            continue
        result = dict(chunk)
        result.update(
            {
                "rank": len(selected) + 1,
                "dense_similarity": round(float(dense_scores[index]), 6),
                "bm25_score": round(float(bm25_scores[index]), 6),
                "hybrid_rrf_score": round(float(hybrid_scores[index]), 9),
            }
        )
        selected.append(result)
        counts[chunk["pmcid"]] += 1
        if len(selected) == top_k:
            break
    return selected

## 7. Screen the candidate final-evaluation questions

The 15 questions contain three questions for each corpus theme and do not repeat the Stage 3 development questions or the Stage 4 demonstration question. This cell saves the first five hybrid results for manual answerability checking.

The automatic theme count is only a screening aid. A question is not frozen until its retrieved passages are inspected and at least three plausible passages from at least two PMC articles are confirmed.

In [ ]:
CANDIDATE_QUESTIONS = [
    {
        "question_id": "M1",
        "theme": "medical_imaging_and_diagnosis",
        "question_type": "applications_and_barriers",
        "question": "How can AI-assisted ultrasound support diagnosis, and what technical and clinical barriers must be addressed before routine use?",
    },
    {
        "question_id": "M2",
        "theme": "medical_imaging_and_diagnosis",
        "question_type": "opportunities_and_risks",
        "question": "What opportunities and risks arise when AI is integrated across radiology, radiation oncology and nuclear medicine workflows?",
    },
    {
        "question_id": "M3",
        "theme": "medical_imaging_and_diagnosis",
        "question_type": "validation_and_monitoring",
        "question": "How should medical-imaging AI be validated and monitored after deployment to preserve accuracy and fairness?",
    },
    {
        "question_id": "C1",
        "theme": "clinical_decision_support",
        "question_type": "personalised_treatment",
        "question": "How can genomic information and medical history be combined by AI to support personalised treatment decisions, and what limitations remain?",
    },
    {
        "question_id": "C2",
        "theme": "clinical_decision_support",
        "question_type": "cancer_decision_support",
        "question": "What role can AI play in cancer diagnosis and treatment planning, and what evidence is needed before clinical adoption?",
    },
    {
        "question_id": "C3",
        "theme": "clinical_decision_support",
        "question_type": "responsibility",
        "question": "How should responsibility be shared between clinicians and AI systems when decision-support recommendations influence patient care?",
    },
    {
        "question_id": "O1",
        "theme": "healthcare_operations",
        "question_type": "resource_use_and_fairness",
        "question": "How can prediction of missed outpatient appointments improve resource use, and what fairness risks must be managed?",
    },
    {
        "question_id": "O2",
        "theme": "healthcare_operations",
        "question_type": "documentation_and_implementation",
        "question": "What benefits and implementation barriers arise when generative AI is used for healthcare documentation and administrative work?",
    },
    {
        "question_id": "O3",
        "theme": "healthcare_operations",
        "question_type": "post_deployment_monitoring",
        "question": "Why can a clinically validated AI system become unreliable after deployment, and what continuing monitoring is required?",
    },
    {
        "question_id": "G1",
        "theme": "generative_ai_and_llms",
        "question_type": "consistency_and_safety",
        "question": "What does inconsistency across repeated responses mean for the safe use of LLMs in medical question answering?",
    },
    {
        "question_id": "G2",
        "theme": "generative_ai_and_llms",
        "question_type": "research_synthesis",
        "question": "How can LLMs support research and information synthesis for rare disorders while limiting hallucination and misinformation?",
    },
    {
        "question_id": "G3",
        "theme": "generative_ai_and_llms",
        "question_type": "safeguards",
        "question": "What safeguards are required when LLMs generate medical research summaries or patient-facing explanations?",
    },
    {
        "question_id": "E1",
        "theme": "ethics_safety_and_bias",
        "question_type": "privacy_fairness_governance",
        "question": "How can federated learning protect healthcare privacy while still raising fairness and governance concerns?",
    },
    {
        "question_id": "E2",
        "theme": "ethics_safety_and_bias",
        "question_type": "bias_pathways",
        "question": "At which stages of development and deployment can bias enter clinical AI, and how can harm to underrepresented groups be reduced?",
    },
    {
        "question_id": "E3",
        "theme": "ethics_safety_and_bias",
        "question_type": "transparency_and_accountability",
        "question": "What transparency and accountability controls are needed when AI generates medical text used in healthcare settings?",
    },
]

question_rows = []
retrieval_rows = []
for item in CANDIDATE_QUESTIONS:
    results = hybrid_retrieve(item["question"])
    expected_theme_hits = sum(
        result["theme"] == item["theme"] for result in results
    )
    unique_articles = len({result["pmcid"] for result in results})
    question_rows.append(
        {
            **item,
            "retrieved_passages": len(results),
            "unique_articles": unique_articles,
            "expected_theme_hits_at_5": expected_theme_hits,
            "automatic_screening_status": (
                "ready_for_manual_review"
                if len(results) >= 3 and unique_articles >= 2
                else "replace_before_experiment"
            ),
            "manual_answerability_decision": "PENDING",
            "manual_notes": "",
        }
    )
    for result in results:
        retrieval_rows.append(
            {
                "question_id": item["question_id"],
                "question_theme": item["theme"],
                "question": item["question"],
                "rank": result["rank"],
                "chunk_id": result["chunk_id"],
                "pmcid": result["pmcid"],
                "article_theme": result["theme"],
                "title": result["title"],
                "source_url": result["source_url"],
                "dense_similarity": result["dense_similarity"],
                "bm25_score": result["bm25_score"],
                "hybrid_rrf_score": result["hybrid_rrf_score"],
                "passage": result["text"],
            }
        )

print("Candidate questions:", len(question_rows))
for row in question_rows:
    print(
        row["question_id"],
        "| sources:", row["unique_articles"],
        "| expected-theme results:", row["expected_theme_hits_at_5"],
    )

## 8. Inspect every candidate question

This compact display helps confirm whether the passages actually answer each question. The full text is also saved in CSV for the formal manual audit.

In [ ]:
for question in CANDIDATE_QUESTIONS:
    print("\n" + "=" * 100)
    print(question["question_id"], question["question"])
    for result in [
        row for row in retrieval_rows
        if row["question_id"] == question["question_id"]
    ]:
        print(
            f'  {result["rank"]}. {result["pmcid"]} | '
            f'{result["title"]} | theme={result["article_theme"]}'
        )
        print("     ", result["passage"][:350].replace("\n", " "), "...")

## 9. Save the Stage 3.2 evidence package

The package contains the cleaned corpus, rebuilt chunks and embeddings, candidate question bank, complete retrieval results, cleaning audit, fixed settings and SHA-256 checksums.

In [ ]:
def write_csv(path, rows, fieldnames=None):
    """Write a list of dictionaries to a UTF-8 CSV file."""
    if not rows:
        raise ValueError(f"No rows available for {path.name}")
    fieldnames = fieldnames or list(rows[0])
    with path.open("w", encoding="utf-8", newline="") as file_handle:
        # Some output rows contain extra internal fields. They are kept in
        # JSONL, while the CSV receives only the columns listed above.
        writer = csv.DictWriter(
            file_handle,
            fieldnames=fieldnames,
            extrasaction="ignore",
        )
        writer.writeheader()
        writer.writerows(rows)


# Save the clean corpus and its original audit evidence.
shutil.copy2(INPUT_DIR / "pmc_corpus_cleaned.jsonl", OUTPUT_DIR)
shutil.copy2(INPUT_DIR / "corpus_cleaning_audit.csv", OUTPUT_DIR)
shutil.copy2(INPUT_DIR / "corpus_cleaning_rules.json", OUTPUT_DIR)

chunks_path = OUTPUT_DIR / "pmc_chunks_cleaned.jsonl"
chunks_path.write_text(
    "\n".join(json.dumps(chunk, ensure_ascii=False) for chunk in chunks) + "\n",
    encoding="utf-8",
)
write_csv(
    OUTPUT_DIR / "pmc_chunk_manifest_cleaned.csv",
    chunks,
    [
        "chunk_id", "pmcid", "doi", "title", "authors", "year",
        "theme", "source_url", "chunk_position", "chunk_word_count",
    ],
)
np.save(OUTPUT_DIR / "pmc_chunk_embeddings_cleaned.npy", embeddings)
write_csv(OUTPUT_DIR / "candidate_question_answerability.csv", question_rows)
write_csv(OUTPUT_DIR / "candidate_question_retrieval_results.csv", retrieval_rows)
(OUTPUT_DIR / "candidate_question_bank.json").write_text(
    json.dumps(CANDIDATE_QUESTIONS, indent=2), encoding="utf-8"
)

metadata = {
    "stage": "Stage 3.2 cleaned corpus and final-question screening",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "article_count": len(records),
    "chunk_count": len(chunks),
    "cleaned_article_count": sum(bool(r.get("cleaning_applied")) for r in records),
    "embedding_model": MODEL_NAME,
    "embedding_model_revision": MODEL_REVISION,
    "sentence_transformers_version": importlib.metadata.version(
        "sentence-transformers"
    ),
    "transformers_version": importlib.metadata.version("transformers"),
    "python_version": sys.version,
    "embedding_dimensions": int(embeddings.shape[1]),
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
    "minimum_final_chunk_words": MINIMUM_FINAL_CHUNK_WORDS,
    "retrieval_method": "weighted reciprocal rank fusion of dense and BM25 ranks",
    "bm25_tokenisation": (
        "Stage 3.1 TOKEN_PATTERN, English stop-word removal and minimum "
        "token length of two"
    ),
    "retrieval_implementation_alignment": (
        "BM25 tokenisation, scoring, RRF and source limiting are copied "
        "from the tested Stage 3.1 implementation."
    ),
    "dense_weight": DENSE_WEIGHT,
    "bm25_weight": BM25_WEIGHT,
    "rrf_constant": RRF_CONSTANT,
    "maximum_per_article": MAXIMUM_PER_ARTICLE,
    "candidate_question_count": len(CANDIDATE_QUESTIONS),
    "question_status": (
        "Candidate questions only. Manual passage-level answerability review "
        "is required before freezing the final experiment."
    ),
}
(OUTPUT_DIR / "stage_3_2_run_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

checksum_rows = {}
for output_file in sorted(OUTPUT_DIR.iterdir()):
    if output_file.is_file() and output_file.name != "checksums_sha256.json":
        checksum_rows[output_file.name] = hashlib.sha256(
            output_file.read_bytes()
        ).hexdigest()
(OUTPUT_DIR / "checksums_sha256.json").write_text(
    json.dumps(checksum_rows, indent=2), encoding="utf-8"
)

archive_path = shutil.make_archive(
    "/content/Adarsh_Konderu_Stage_3_2_Clean_Retrieval_Evidence_v2",
    "zip",
    OUTPUT_DIR,
)
print("Saved chunks:", len(chunks))
print("Evidence package:", archive_path)
files.download(archive_path)

## 10. What this stage demonstrates

- The original 46-article inclusion decision is preserved.
- Confirmed publication-system artefacts are removed using documented rules.
- Document and query embeddings use the same pinned model revision and package version.
- The selected hybrid retrieval settings are unchanged.
- Candidate evaluation questions are screened before any A/B/C report is generated.
- Every cleaned record, chunk, vector, question and result can be traced through saved files and checksums.

The next action is to inspect the downloaded question-result CSV and freeze only answerable questions. No experimental report should be generated before that decision is recorded.